# extrator universal de documentos (pdf, docx)

este notebook permite extrair texto e tabelas de arquivos pdf e docx.

**funcionalidades:**
1.  **upload de arquivo:** envie seu arquivo (pdf, docx, etc.).
2.  **configuração de saída:** escolha como salvar as tabelas no excel.
3.  **feature a (foco em pdf):** extrai tabelas de pdfs usando `tabula-py`.
4.  **feature b (universal):** extrai texto e tabelas de `unstructured`.
5.  **feature c (pdf - empilhamento):** extrai tabelas de pdf e "empilha" continuações (lógica `pdfplumber`).

## 1. instalação das dependências

primeiro, executamos a instalação das bibliotecas necessárias.

In [ ]:
# feature b: biblioteca principal para extração de texto e layout
!pip install "unstructured[pdf,docx]" -q

# feature a: biblioteca especialista em extração de tabelas de pdf
!pip install tabula-py -q

# feature c: biblioteca alternativa para pdf e empilhamento
!pip install pdfplumber -q

# biblioteca para manipulação e visualização das tabelas (a, b, c)
!pip install pandas -q

# biblioteca necessária para o pandas escrever arquivos .xlsx (excel) (a, b, c)
!pip install openpyxl -q

# 'unstructured' (e 'tabula') podem precisar de ferramentas do sistema para ler pdfs
# primeiro, atualiza a lista de pacotes do sistema (corrige o erro 404 not found)
!sudo apt-get update -q
# agora, instala as dependências
!sudo apt-get install -y poppler-utils tesseract-ocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://packages.cloud.google.com/apt gcsfuse-jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Readin

## 2. importação das bibliotecas

agora, importamos os módulos que vamos usar.

In [ ]:
import os
import pandas as pd
import io
from google.colab import files
from IPython.display import display, HTML, Markdown

# imports da feature a
import tabula

# imports da feature b
from unstructured.partition.auto import partition
from unstructured.documents.elements import Table

# imports da feature c
import pdfplumber

print("bibliotecas carregadas com sucesso.")

bibliotecas carregadas com sucesso.


## 3. upload do arquivo

execute a célula abaixo para selecionar o arquivo (pdf ou docx) do seu computador.

In [ ]:
uploaded = files.upload()

if not uploaded:
  print("nenhum arquivo foi enviado.")
else:
  # salva o nome do arquivo para uso posterior
  input_filepath = list(uploaded.keys())[0]
  print(f"\narquivo '{input_filepath}' carregado com sucesso.")

  # salva o arquivo em disco no ambiente do colab
  with open(input_filepath, 'wb') as f:
      f.write(uploaded[input_filepath])

Saving PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf to PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf

arquivo 'PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf' carregado com sucesso.


## 4. configuração da saída

defina como as tabelas devem ser salvas no arquivo excel (apenas para features a e b).

In [ ]:
# mude para true se quiser salvar todas as tabelas em uma única aba (uma embaixo da outra)
# mude para false se quiser que cada tabela fique em sua própria aba (ex: "tabela_1", "tabela_2")
salvar_em_uma_aba = False

---

## feature a: extração rápida de tabelas (foco em pdf)

esta feature usa `tabula-py`, ideal para extrair tabelas de arquivos **pdf**.

In [ ]:
if 'input_filepath' not in locals() or not os.path.exists(input_filepath):
    print("erro: faça o upload de um arquivo na célula '3. upload do arquivo' primeiro.")
elif not input_filepath.lower().endswith('.pdf'):
    print(f"a feature a (tabula) funciona melhor com pdfs. pulando arquivo: {input_filepath}")
else:
    print(f"(feature a) procurando tabelas em '{input_filepath}' com tabula...")
    try:
        # tenta ler todas as tabelas de todas as páginas
        tables_tabula = tabula.read_pdf(input_filepath, pages='all', multiple_tables=True)

        if not tables_tabula:
            print("(feature a) tabula não encontrou tabelas neste pdf.")
        else:
            print(f"(feature a) encontradas {len(tables_tabula)} tabelas!")

            excel_path = 'tabelas_extraidas_tabula.xlsx'

            if salvar_em_uma_aba:
                print("(feature a) salvando todas as tabelas em uma única aba ('todas_tabelas')...")
                with pd.ExcelWriter(excel_path) as writer:
                    current_row = 0
                    for i, table_df in enumerate(tables_tabula):
                        display(Markdown(f"**tabela {i+1} (tabula)**"))
                        display(table_df)

                        # escreve o nome da tabela
                        pd.DataFrame([f"tabela {i+1}"]).to_excel(writer, sheet_name='todas_tabelas', startrow=current_row, index=False, header=False)
                        current_row += 1

                        # escreve a tabela
                        table_df.to_excel(writer, sheet_name='todas_tabelas', startrow=current_row, index=False)
                        current_row += len(table_df) + 2 # +2 para uma linha de espaço
            else:
                print("(feature a) salvando cada tabela em uma aba separada...")
                with pd.ExcelWriter(excel_path) as writer:
                    for i, table_df in enumerate(tables_tabula):
                        display(Markdown(f"**tabela {i+1} (tabula)**"))
                        display(table_df) # mostra a tabela no notebook
                        table_df.to_excel(writer, sheet_name=f'tabela_{i+1}', index=False)

            print(f"\n(feature a) tabelas salvas em '{excel_path}'.")
            # oferece o download do arquivo excel
            files.download(excel_path)

    except Exception as e:
        print(f"(feature a) ocorreu um erro ao processar o pdf com tabula: {e}")

(feature a) procurando tabelas em 'PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf' com tabula...
(feature a) encontradas 70 tabelas!
(feature a) salvando cada tabela em uma aba separada...


**tabela 1 (tabula)**

,.,.Código IBGE,Unnamed: 0,.UF,Unnamed: 1,Unnamed: 2,.Ente,.Rede,.Valor total do,fomento,...,em,.Valor de.1,repasse.1,Unnamed: 6,em.1,.Valor de.2,repasse.2,Unnamed: 7,em.2,janeiro
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,repassado,NaN,...,NaN,dezembro,de,2025.0,NaN,NaN,de 2026,NaN,NaN,NaN
1,.,0.120000,NaN,.AC,.Acre,NaN,NaN,.Estadual,". R$ 11.546.264,40",NaN,...,NaN,". R$ 2.453.581,18",NaN,NaN,NaN,". R$ 1.731.939,67",NaN,NaN,NaN,NaN
2,.,0.120001,NaN,.AC,.Acrelândia,NaN,NaN,.Municipal,". R$ 399.600,51",NaN,...,NaN,". R$ 84.915,10",NaN,NaN,NaN,". R$ 59.940,10",NaN,NaN,NaN,NaN
3,.,0.120005,NaN,.AC,.Assis Brasil,NaN,NaN,.Municipal,". R$ 345.417,39",NaN,...,NaN,". R$ 73.401,19",NaN,NaN,NaN,". R$ 51.812,62",NaN,NaN,NaN,NaN
4,.,0.120010,NaN,.AC,.Brasiléia,NaN,NaN,.Municipal,". R$ 738.245,01",NaN,...,NaN,". R$ 156.877,06",NaN,NaN,NaN,". R$ 110.736,77",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,.,0.270461,NaN,.AL,.Maravilha,NaN,NaN,.Municipal,". R$ 203.186,70",NaN,...,NaN,". R$ 43.177,17",NaN,NaN,NaN,". R$ 30.478,02",NaN,NaN,NaN,NaN
69,.,0.270471,NaN,.AL,.Marechal Deodoro,NaN,NaN,.Municipal,". R$ 1.835.453,19",NaN,...,NaN,". R$ 390.033,80",NaN,NaN,NaN,". R$ 275.317,99",NaN,NaN,NaN,NaN
70,.,0.270481,NaN,.AL,.Maribondo,NaN,NaN,.Municipal,". R$ 203.186,70",NaN,...,NaN,". R$ 43.177,17",NaN,NaN,NaN,". R$ 30.478,02",NaN,NaN,NaN,NaN
71,.,0.270510,NaN,.AL,.Matriz de Camaragibe,NaN,NaN,.Municipal,". R$ 907.567,26",NaN,...,NaN,". R$ 192.858,04",NaN,NaN,NaN,". R$ 136.135,10",NaN,NaN,NaN,NaN


**tabela 2 (tabula)**

,.,.2705507,.AL,.Murici,Unnamed: 0,Unnamed: 1,.Municipal,..1,R$,"907.567,26",...,"385.716,08",..3,R$.2,"192.858,04",..4,R$.3,"192.858,04.1",..5,R$.4,"136.135,10"
0,.,0.270570,.AL,.Olho d'Água das Flores,NaN,NaN,.Municipal,.,R$,"839.838,36",...,"356.931,30",.,R$,"178.465,65",.,R$,"178.465,65",.,R$,"125.975,76"
1,.,0.270580,.AL,.Olho d'Água do Casado,NaN,NaN,.Municipal,.,R$,"293.943,42",...,"124.925,95",.,R$,"62.462,97",.,R$,"62.462,97",.,R$,"44.091,53"
2,.,0.270590,.AL,.Olho d'Água Grande,NaN,NaN,.Municipal,.,R$,"203.186,70",...,"86.354,34",.,R$,"43.177,17",.,R$,"43.177,17",.,R$,"30.478,02"
3,.,0.270600,.AL,.Olivença,NaN,NaN,.Municipal,.,R$,"507.966,75",...,"215.885,86",.,R$,"107.942,93",.,R$,"107.942,93",.,R$,"76.195,03"
4,.,0.270621,.AL,.Palestina,NaN,NaN,.Municipal,.,R$,"230.278,26",...,"97.868,26",.,R$,"48.934,13",.,R$,"48.934,13",.,R$,"34.541,74"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,.,0.130165,.AM,.Guajará,NaN,NaN,.Municipal,.,R$,"501.193,86",...,"213.007,39",.,R$,"106.503,69",.,R$,"106.503,69",.,R$,"75.179,09"
57,.,0.130170,.AM,.Humaitá,NaN,NaN,.Municipal,.,R$,"1.584.856,26",...,"673.563,91",.,R$,"336.781,95",.,R$,"336.781,95",.,R$,"237.728,45"
58,.,0.130180,.AM,.Ipixuna,NaN,NaN,.Municipal,.,R$,"643.424,55",...,"273.455,43",.,R$,"136.727,71",.,R$,"136.727,71",.,R$,"96.513,70"
59,.,0.130185,.AM,.Iranduba,NaN,NaN,.Municipal,.,R$,"2.139.555,95",...,"909.311,27",.,R$,"454.655,63",.,R$,"454.655,63",.,R$,"320.933,42"


...

**tabela 70 (tabula)**

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN



(feature a) tabelas salvas em 'tabelas_extraidas_tabula.xlsx'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## feature b: extração inteligente (universal - pdf, docx)

esta feature usa `unstructured` para analisar o layout do documento. funciona para **pdfs e docx**.

In [ ]:
if 'input_filepath' not in locals() or not os.path.exists(input_filepath):
    print("erro: faça o upload de um arquivo na célula '3. upload do arquivo' primeiro.")
else:
    print(f"(feature b) processando '{input_filepath}' com unstructured (strategy='auto')...")

    # strategy='auto' decide a melhor forma de extração
    # (incluindo ocr se necessário, graças ao tesseract que instalamos)
    elements = partition(filename=input_filepath, strategy="auto")

    # 1. separa o texto comum
    text_elements = [el for el in elements if not isinstance(el, Table)]
    full_text = "\n\n".join([str(el) for el in text_elements])

    if full_text:
        print("(feature b) --- início do texto extraído (amostra) ---")
        print(full_text[:1500] + "...")
        print("(feature b) --- fim da amostra ---\n")

        # salva o texto em .txt e oferece download
        text_output_path = "texto_extraido_unstructured.txt"
        with open(text_output_path, 'w', encoding='utf-8') as f:
            f.write(full_text)
        print(f"(feature b) texto completo salvo em '{text_output_path}'.")
        files.download(text_output_path)
    else:
        print("(feature b) nenhum texto comum foi extraído.")

    # 2. separa as tabelas
    table_elements = [el for el in elements if isinstance(el, Table)]

    if not table_elements:
        print("\n(feature b) unstructured não encontrou tabelas neste documento.")
    else:
        print(f"\n(feature b) encontradas {len(table_elements)} tabelas!")

        excel_path_uns = 'tabelas_extraidas_unstructured.xlsx'

        # aqui processamos os dataframes primeiro
        dataframes_extraidos = []
        for i, table_el in enumerate(table_elements):
            display(Markdown(f"**tabela {i+1} (unstructured)**"))
            try:
                # unstructured extrai tabelas como html. nós convertemos para dataframe.
                table_html = table_el.metadata.text_as_html
                # pd.read_html retorna uma *lista* de dataframes
                table_df = pd.read_html(io.StringIO(table_html))[0]
                dataframes_extraidos.append(table_df)
                display(table_df)
            except Exception as e:
                print(f"(feature b) erro ao converter tabela {i+1}: {e}")
                # às vezes, o html é a melhor visualização
                display(HTML(table_html))

        # agora salvamos, com base na escolha do usuário
        if salvar_em_uma_aba:
            print("(feature b) salvando todas as tabelas em uma única aba ('todas_tabelas')...")
            with pd.ExcelWriter(excel_path_uns) as writer:
                current_row = 0
                for i, table_df in enumerate(dataframes_extraidos):
                    # escreve o nome da tabela
                    pd.DataFrame([f"tabela {i+1}"]).to_excel(writer, sheet_name='todas_tabelas', startrow=current_row, index=False, header=False)
                    current_row += 1
                    # escreve a tabela
                    table_df.to_excel(writer, sheet_name='todas_tabelas', startrow=current_row, index=False)
                    current_row += len(table_df) + 2 # +2 para uma linha de espaço
        else:
            print("(feature b) salvando cada tabela em uma aba separada...")
            with pd.ExcelWriter(excel_path_uns) as writer:
                for i, table_df in enumerate(dataframes_extraidos):
                    table_df.to_excel(writer, sheet_name=f'tabela_{i+1}', index=False)

        print(f"\n(feature b) todas as tabelas foram salvas em '{excel_path_uns}'.")
        files.download(excel_path_uns)


(feature b) processando 'PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf' com unstructured (strategy='auto')...
(feature b) --- início do texto extraído (amostra) ---
Documento assinado digitalmente conforme MP nº 2.200-2 de 24/08/2001,que institui a Infraestrutura de Chaves Públicas Brasileira - ICP-Brasil.Este documento pode ser verificado no endereço eletrônicohttp://www.in.gov.br/autenticidade.html, pelo código 0515202510020005252

Nº 188, quinta-feira, 2 de outubro de 2025ISSN 1677-7042Seção 1

Ministério da EducaçãoGABINETE DO MINISTROPORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025AlteraaPortaria MECnº605,de 29deagostode2025, quedispõesobreas diretrizes para acriação de matrículas em tempo integral, na educação básica, no âmbito do Fundo de ManutençãoeDesenvolvimentodaEducaçãoBásicae deValorizaçãodosProfissionaisdaEducação -Fundeb.O MINISTRO DE ESTADO DA EDUCAÇÃO, no uso das atribuições que lhe confere o art. 87, parágrafo único, incisos II e IV, da Constituição, e tendo em

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


(feature b) unstructured não encontrou tabelas neste documento.


---

##extração com empilhamento

esta feature usa `pdfplumber` para "empilhar" tabelas que são continuações umas das outras. o resultado é salvo em **uma única aba**.

In [ ]:
# funções de ajuda da feature c

def clean_cell_c(cell_value):
    """
    limpa o lixo de uma célula extraída pelo pdfplumber.
    (remove espaços em branco e os pontos extras no início/fim)
    """
    if cell_value is None:
        return None

    # remove espaços e os pontos extras
    cleaned = str(cell_value).strip().strip('.')

    # se a célula era apenas um '.' ou ' ', retorna none
    if cleaned == "":
        return None
    return cleaned


def extract_and_stack_tables(filepath, output_path):
    """
    extrai todas as tabelas, limpa-as e empilha (concatena)
    aquelas que parecem ser parte da mesma tabela lógica.
    """
    print(f"(feature c) processando e empilhando tabelas de: {filepath}...")

    all_data_rows = []
    master_header = None
    header_col_count = 0

    with pdfplumber.open(filepath) as pdf:

        for i, page in enumerate(pdf.pages):
            # extrai todas as tabelas da página
            tables = page.extract_tables()

            for j, table_data in enumerate(tables):
                if not table_data:
                    continue # pula se a tabela estiver vazia

                # --- 1. limpeza (corrige os 'pontos extras') ---
                cleaned_table = []
                for row in table_data:
                    cleaned_row = [clean_cell_c(cell) for cell in row]
                    # adiciona a linha só se ela não ficou completamente vazia
                    if any(cleaned_row):
                        cleaned_table.append(cleaned_row)

                if not cleaned_table:
                    continue # pula se a tabela limpa estiver vazia

                # --- 2. empilhamento (corrige as 'milhares de abas') ---

                # se o cabeçalho mestre ainda não foi definido, use esta tabela
                if master_header is None:
                    master_header = cleaned_table[0]
                    header_col_count = len(master_header)
                    all_data_rows.extend(cleaned_table[1:]) # adiciona os dados
                    print(f"(feature c) cabeçalho mestre definido (p{i+1}_t{j+1}) com {header_col_count} colunas.")

                # se já temos um cabeçalho, verifique se esta tabela é uma continuação
                else:
                    # se a primeira linha desta tabela não é o cabeçalho,
                    # mas a contagem de colunas bate, assuma que é dados.
                    if cleaned_table[0] != master_header and len(cleaned_table[0]) == header_col_count:
                        all_data_rows.extend(cleaned_table) # adiciona todas as linhas

                    # se o cabeçalho for repetido, adicione apenas os dados
                    elif cleaned_table[0] == master_header:
                        all_data_rows.extend(cleaned_table[1:])

                    else:
                        print(f"  -> (feature c) ignorando tabela p{i+1}_t{j+1} (cabeçalho/colunas incompatíveis)")

    if not all_data_rows or master_header is None:
        print("(feature c) nenhum dado de tabela foi encontrado ou o cabeçalho não foi definido.")
        return 0

    print(f"(feature c) concatenando {len(all_data_rows)} linhas de dados em uma única tabela...")

    # cria o dataframe final com todas as linhas e o cabeçalho mestre
    final_df = pd.DataFrame(all_data_rows, columns=master_header)

    # remove quaisquer linhas que possam estar totalmente vazias
    final_df.dropna(how='all', inplace=True)

    print(f"(feature c) dataframe final criado com {len(final_df)} linhas.")

    # salva em uma única aba
    # o engine 'openpyxl' salva em utf-8 por padrão, o excel vai ler corretamente.
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        final_df.to_excel(writer, sheet_name="tabelas_empilhadas", index=False)

    return len(final_df)


### execução (feature c)

In [ ]:
if 'input_filepath' not in locals() or not os.path.exists(input_filepath):
    print("erro: faça o upload de um arquivo na célula '3. upload do arquivo' primeiro.")
elif not input_filepath.lower().endswith('.pdf'):
    print(f"a feature c (pdfplumber) funciona apenas com pdfs. pulando arquivo: {input_filepath}")
else:
    output_file_c = "tabelas_empilhadas_pdfplumber.xlsx"

    try:
        # 1. extrair, limpar e empilhar
        total_linhas = extract_and_stack_tables(input_filepath, output_file_c)

        print(f"\n--- (feature c) sucesso ---")
        print(f"extração e limpeza concluídas.")
        print(f"{total_linhas} linhas salvas em uma única aba.")
        print(f"baixando resultado: {output_file_c}")

        # 4. fazer o download do arquivo final
        files.download(output_file_c)

    except Exception as e:
        print(f"\n--- (feature c) erro inesperado ---")
        print(f"ocorreu uma falha durante a extração:")
        print(e)

(feature c) processando e empilhando tabelas de: PORTARIA MEC Nº 669, DE 1º DE OUTUBRO DE 2025 1 (1).pdf...
(feature c) cabeçalho mestre definido (p1_t1) com 9 colunas.
(feature c) concatenando 4076 linhas de dados em uma única tabela...
(feature c) dataframe final criado com 4076 linhas.

--- (feature c) sucesso ---
extração e limpeza concluídas.
4076 linhas salvas em uma única aba.
baixando resultado: tabelas_empilhadas_pdfplumber.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>